In [1]:
!pip install openai sentence-transformers -q

In [2]:
import json
from datetime import datetime
from getpass import getpass
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import numpy as np

groq_api_key = getpass("Pegá tu Groq API Key: ")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

modelo_embeddings = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Setup listo")

Pegá tu Groq API Key: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Setup listo


In [3]:
faq_database = [
    {
        "respuesta": "El plazo fijo Helix Premium ofrece una tasa de 118% TNA para montos mayores a $500.000.",
        "variantes": ["tasa del plazo fijo", "cuánto rinde un plazo fijo en Helix", "interés si pongo plata a plazo fijo"]
    },
    {
        "respuesta": "El FCI de renta fija tiene un rendimiento promedio de 9.2% mensual y rescate en 24-48hs.",
        "variantes": ["rendimiento del FCI de renta fija", "cuánto gano si dejo mi plata quieta en un fondo", "rentabilidad mensual del fondo"]
    },
    {
        "respuesta": "El FCI Money Market permite rescate inmediato el mismo día hábil, ideal para liquidez.",
        "variantes": ["necesito la plata ya no puedo esperar", "fondo con liquidez inmediata", "quiero retirar mi dinero el mismo día"]
    },
    {
        "respuesta": "Para abrir una cuenta Helix Black necesitás un patrimonio mínimo de $5.000.000 o ingresos de $500.000 mensuales.",
        "variantes": ["requisitos para la cuenta Helix Black", "cuánto necesito para ser cliente premium"]
    },
    {
        "respuesta": "Podés operar dólar MEP y cable a través de la mesa de cambio de Helix.",
        "variantes": ["cómo compro dólares en Helix", "opciones para dolarizarme"]
    },
    {
        "respuesta": "La tarjeta Mastercard Black no tiene costo de mantenimiento para clientes Helix Black.",
        "variantes": ["la tarjeta black tiene costo de mantenimiento", "cuánto pago por la tarjeta premium"]
    },
    {
        "respuesta": "El horario de atención del equipo de soporte es de lunes a viernes de 9 a 18hs.",
        "variantes": ["a qué hora puedo llamar si tengo un problema", "horario del soporte al cliente"]
    },
    {
        "respuesta": "Para cancelar un plazo fijo antes de la fecha de vencimiento, contactá a tu asesor — puede haber penalidades.",
        "variantes": ["puedo retirar el plazo fijo antes de tiempo", "qué pasa si cancelo el plazo fijo antes"]
    },
    {
        "respuesta": "Helix no cobra comisión por transferencias entre cuentas propias.",
        "variantes": ["hay algún costo por mover dinero entre mis cuentas", "cobran comisión por transferir entre mis cuentas"]
    },
    {
        "respuesta": "Los fondos comunes de inversión no tienen garantía del Estado, a diferencia de los plazos fijos bancarios tradicionales.",
        "variantes": ["el gobierno me devuelve la plata si el fondo quiebra", "los fondos comunes están garantizados por el estado"]
    },
    {
        "respuesta": "Podés consultar el estado de tus inversiones las 24 horas desde la app de Helix.",
        "variantes": ["puedo ver mis inversiones desde el celular", "la app funciona las 24 horas"]
    }
]

# Indexamos las VARIANTES (preguntas), mapeadas a sus respuestas
todas_variantes, indice_respuesta = [], []
for entry in faq_database:
    for variante in entry["variantes"]:
        todas_variantes.append(variante)
        indice_respuesta.append(entry["respuesta"])

embeddings_variantes = modelo_embeddings.encode(todas_variantes)


def similitud_coseno(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def buscar_faq(consulta, umbral=0.6):
    """Busca la FAQ más relevante. Devuelve (respuesta, score) o (None, score) si no supera el umbral."""
    embedding_consulta = modelo_embeddings.encode(consulta)
    similitudes = [
        (similitud_coseno(embedding_consulta, emb), indice_respuesta[i])
        for i, emb in enumerate(embeddings_variantes)
    ]
    similitudes.sort(key=lambda x: x[0], reverse=True)
    score, respuesta = similitudes[0]
    if score >= umbral:
        return respuesta, score
    return None, score


print(f"✅ {len(todas_variantes)} variantes indexadas de {len(faq_database)} FAQs")

✅ 25 variantes indexadas de 11 FAQs


In [4]:
def consultar_saldo(cliente_id: str) -> str:
    saldos = {
        "C001": {"saldo_ars": 1500000, "saldo_usd": 8000},
        "C002": {"saldo_ars": 320000, "saldo_usd": 1200},
    }
    datos = saldos.get(cliente_id)
    if not datos:
        return f"No se encontró el cliente {cliente_id}"
    return f"Saldo cliente {cliente_id}: ${datos['saldo_ars']:,} ARS y USD {datos['saldo_usd']:,}"


def listar_movimientos(cliente_id: str, cantidad: int = 3) -> str:
    movimientos = {
        "C001": [
            "Transferencia recibida: +$200.000 (10/06/2026)",
            "Pago de servicios: -$45.000 (08/06/2026)",
            "Compra USD: -$300.000 / +USD 500 (05/06/2026)",
        ],
        "C002": [
            "Acreditación de sueldo: +$320.000 (01/06/2026)",
            "Transferencia enviada: -$50.000 (03/06/2026)",
        ],
    }
    movs = movimientos.get(cliente_id, [])
    if not movs:
        return f"No hay movimientos para el cliente {cliente_id}"
    return "\n".join(movs[:cantidad])


def buscar_producto(nombre_producto: str) -> str:
    """Busca info de un producto. Catálogo ampliado con sinónimos/variantes genéricas."""
    catalogo = {
        "plazo fijo": "Plazo Fijo Helix Premium: 118% TNA, plazos de 30 a 365 días.",
        "fci renta fija": "FCI Renta Fija Helix: rendimiento ~9.2% mensual, rescate en 24-48hs.",
        "fci money market": "FCI Money Market Helix: rescate inmediato, ideal para liquidez.",
        # Entradas genéricas — el problema detectado en Ej3
        "fci": "Helix ofrece dos FCI: Renta Fija (rendimiento ~9.2% mensual, rescate 24-48hs) y "
               "Money Market (rescate inmediato, ideal para liquidez).",
        "fondo": "Helix ofrece dos FCI: Renta Fija (rendimiento ~9.2% mensual, rescate 24-48hs) y "
                 "Money Market (rescate inmediato, ideal para liquidez).",
    }
    nombre_lower = nombre_producto.lower()
    # Buscar coincidencia exacta primero, luego parcial
    if nombre_lower in catalogo:
        return catalogo[nombre_lower]
    for key, value in catalogo.items():
        if key in nombre_lower or nombre_lower in key:
            return value
    return "Producto no encontrado en el catálogo."


funciones_disponibles = {
    "consultar_saldo": consultar_saldo,
    "listar_movimientos": listar_movimientos,
    "buscar_producto": buscar_producto,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "consultar_saldo",
            "description": "Consulta el saldo disponible (ARS y USD) de un cliente por su ID. Usar SOLO cuando el cliente pide su saldo específico, no para preguntas generales sobre productos.",
            "parameters": {
                "type": "object",
                "properties": {"cliente_id": {"type": "string", "description": "ID del cliente, formato 'C001'."}},
                "required": ["cliente_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "listar_movimientos",
            "description": "Devuelve los últimos movimientos de la cuenta de un cliente. Usar SOLO cuando el cliente pide ver sus movimientos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "cliente_id": {"type": "string", "description": "ID del cliente, formato 'C001'."},
                    "cantidad": {"type": "integer", "description": "Cantidad de movimientos. Default 3."}
                },
                "required": ["cliente_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_producto",
            "description": "Busca condiciones EXACTAS y actualizadas de un producto financiero de Helix (tasas, plazos, montos). Usar cuando el cliente pide datos concretos de un producto específico.",
            "parameters": {
                "type": "object",
                "properties": {"nombre_producto": {"type": "string", "description": "Nombre del producto, ej: 'plazo fijo', 'fci'."}},
                "required": ["nombre_producto"]
            }
        }
    }
]

print("✅ Funciones y tools definidos")

✅ Funciones y tools definidos


In [11]:
def asistente_helix(mensaje_usuario, historial, cliente_id="C001", log_sesion=None):
    registro = {
        "timestamp": datetime.now().isoformat(),
        "cliente_id": cliente_id,
        "mensaje_usuario": mensaje_usuario,
        "faq_match": None,
        "faq_score": None,
        "tools_llamadas": [],
        "derivado": False,
    }

    # 1. Búsqueda en FAQs (Ej4) — RAG simple: si hay match, se inyecta como contexto
    faq_respuesta, faq_score = buscar_faq(mensaje_usuario)
    registro["faq_score"] = round(float(faq_score), 3)

    contexto_faq = ""
    if faq_respuesta:
        registro["faq_match"] = faq_respuesta
        contexto_faq = f"\n\n[Info de la base de conocimiento Helix: {faq_respuesta}]"

    # 2. Inicializar historial con system prompt (solo la primera vez)
    if not historial:
        historial.append({
            "role": "system",
            "content": f"""Sos AsesorAI, el asistente virtual de Helix, una fintech digital premium argentina.
            Helix es una fintech, no un banco tradicional: nunca uses esa palabra.
            El cliente que está consultando tiene el ID '{cliente_id}'.
            Sos profesional, conciso y orientado a soluciones.
            Si la consulta está fuera de tu alcance (temas legales, fiscales, reclamos formales,
            o no tenés información suficiente), decilo y ofrecé derivar al asesor de cuenta.
            Nunca inventes tasas, números o condiciones de productos."""
        })

    historial.append({"role": "user", "content": mensaje_usuario + contexto_faq})

    # 3. Request con tools (Ej3)
    respuesta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=historial,
        tools=tools,
        tool_choice="auto",
        temperature=0.4
    )
    mensaje_modelo = respuesta.choices[0].message

    if mensaje_modelo.tool_calls:
        historial.append(mensaje_modelo)
        for tool_call in mensaje_modelo.tool_calls:
            nombre_funcion = tool_call.function.name
            argumentos = json.loads(tool_call.function.arguments)
            registro["tools_llamadas"].append({"funcion": nombre_funcion, "argumentos": argumentos})

            resultado = funciones_disponibles[nombre_funcion](**argumentos)
            historial.append({
                "role": "tool", "tool_call_id": tool_call.id,
                "name": nombre_funcion, "content": resultado
            })

        respuesta_final = client.chat.completions.create(
            model="llama-3.3-70b-versatile", messages=historial, temperature=0.4
        )
        texto_respuesta = respuesta_final.choices[0].message.content
    else:
        texto_respuesta = mensaje_modelo.content

    historial.append({"role": "assistant", "content": texto_respuesta})

    # 4. Detección de derivación REAL (frases de incapacidad explícita, no cortesía/upsell)
    frases_derivacion_real = [
        "no puedo ayudarte con", "no puedo resolver", "está fuera de mi alcance",
        "no tengo información suficiente", "no puedo asistirte con"
    ]
    registro["derivado"] = any(p in texto_respuesta.lower() for p in frases_derivacion_real)
    registro["respuesta_asistente"] = texto_respuesta

    if log_sesion is not None:
        log_sesion.append(registro)

    return texto_respuesta


print("✅ Asistente Helix listo (v2 — derivación corregida)")

✅ Asistente Helix listo (v2 — derivación corregida)


In [7]:
print("CLIENTE:", "Buenísimo. Por otro lado, ¿cuál es mi saldo actual?")
print("\nASESORAI:", asistente_helix(
    "Buenísimo. Por otro lado, ¿cuál es mi saldo actual?",
    historial_a, cliente_id="C001", log_sesion=log_sesion
))

CLIENTE: Buenísimo. Por otro lado, ¿cuál es mi saldo actual?

ASESORAI: Tu saldo actual es de $1.500.000 en pesos argentinos y $8.000 en dólares estadounidenses. ¿Necesitas ayuda con algo más, Carlos? ¿Quieres invertir parte de ese saldo en nuestro FCI de renta fija o tienes alguna otra consulta?


In [8]:
print("CLIENTE:", "¿Y si en cambio necesito algo con liquidez inmediata, por si tengo que usar la plata de un día para el otro?")
print("\nASESORAI:", asistente_helix(
    "¿Y si en cambio necesito algo con liquidez inmediata, por si tengo que usar la plata de un día para el otro?",
    historial_a, cliente_id="C001", log_sesion=log_sesion
))

CLIENTE: ¿Y si en cambio necesito algo con liquidez inmediata, por si tengo que usar la plata de un día para el otro?

ASESORAI: Si necesitas liquidez inmediata, te recomiendo considerar nuestro FCI Money Market. Este producto ofrece la ventaja de poder rescatar tus fondos el mismo día hábil, lo que te permite tener acceso a tu dinero cuando lo necesites. Es ideal para aquellos que requieren flexibilidad y liquidez en sus inversiones. ¿Te gustaría saber más sobre cómo funciona o cómo puedes invertir en este producto, Carlos?


In [9]:
historial_b = []

print("CLIENTE:", "Quiero hacer un reclamo formal: me cobraron una comisión indebida hace 6 meses y todavía no me la devolvieron.")
print("\nASESORAI:", asistente_helix(
    "Quiero hacer un reclamo formal: me cobraron una comisión indebida hace 6 meses y todavía no me la devolvieron.",
    historial_b, cliente_id="C002", log_sesion=log_sesion
))

CLIENTE: Quiero hacer un reclamo formal: me cobraron una comisión indebida hace 6 meses y todavía no me la devolvieron.

ASESORAI: Lo siento, pero no puedo ayudarte con reclamos formales. Te recomiendo que te comuniques con tu asesor de cuenta para que pueda asistirte con tu solicitud. ¿Quieres que te proporcione la información de contacto de tu asesor de cuenta?


In [10]:
for i, registro in enumerate(log_sesion, 1):
    print(f"--- Registro {i} ---")
    print(f"Cliente: {registro['cliente_id']}")
    print(f"Mensaje: {registro['mensaje_usuario'][:60]}...")
    print(f"FAQ match: {'Sí (' + str(registro['faq_score']) + ')' if registro['faq_match'] else 'No (' + str(registro['faq_score']) + ')'}")
    print(f"Tools llamadas: {[t['funcion'] for t in registro['tools_llamadas']] or 'ninguna'}")
    print(f"Derivado: {registro['derivado']}")
    print()

# Guardar el log completo en JSON
with open("conversaciones_log.json", "w", encoding="utf-8") as f:
    json.dump(log_sesion, f, ensure_ascii=False, indent=2)

print("✅ Log guardado en conversaciones_log.json")

--- Registro 1 ---
Cliente: C001
Mensaje: Hola, soy Carlos. Tengo $2.000.000 para invertir, ¿cuánto ri...
FAQ match: Sí (0.665)
Tools llamadas: ['buscar_producto']
Derivado: True

--- Registro 2 ---
Cliente: C001
Mensaje: Buenísimo. Por otro lado, ¿cuál es mi saldo actual?...
FAQ match: Sí (0.628)
Tools llamadas: ['consultar_saldo']
Derivado: False

--- Registro 3 ---
Cliente: C001
Mensaje: ¿Y si en cambio necesito algo con liquidez inmediata, por si...
FAQ match: Sí (0.734)
Tools llamadas: ['buscar_producto']
Derivado: False

--- Registro 4 ---
Cliente: C002
Mensaje: Quiero hacer un reclamo formal: me cobraron una comisión ind...
FAQ match: No (0.531)
Tools llamadas: ninguna
Derivado: True

✅ Log guardado en conversaciones_log.json


In [14]:
historial_a = []
historial_b = []
log_sesion = []

# Conversación A — FAQ + memoria + function calling
print("=" * 60)
print("CONVERSACIÓN A — Cliente C001 (Carlos)")
print("=" * 60)

print("\nCLIENTE:", "Hola, soy Carlos. Tengo $2.000.000 para invertir, ¿cuánto rinde el FCI de renta fija?")
print("ASESORAI:", asistente_helix(
    "Hola, soy Carlos. Tengo $2.000.000 para invertir, ¿cuánto rinde el FCI de renta fija?",
    historial_a, cliente_id="C001", log_sesion=log_sesion
))

print("\nCLIENTE:", "Buenísimo. Por otro lado, ¿cuál es mi saldo actual?")
print("ASESORAI:", asistente_helix(
    "Buenísimo. Por otro lado, ¿cuál es mi saldo actual?",
    historial_a, cliente_id="C001", log_sesion=log_sesion
))

print("\nCLIENTE:", "¿Y si en cambio necesito algo con liquidez inmediata, por si tengo que usar la plata de un día para el otro?")
print("ASESORAI:", asistente_helix(
    "¿Y si en cambio necesito algo con liquidez inmediata, por si tengo que usar la plata de un día para el otro?",
    historial_a, cliente_id="C001", log_sesion=log_sesion
))

# Conversación B — derivación real
print("\n" + "=" * 60)
print("CONVERSACIÓN B — Cliente C002 (reclamo)")
print("=" * 60)

print("\nCLIENTE:", "Quiero hacer un reclamo formal: me cobraron una comisión indebida hace 6 meses y todavía no me la devolvieron.")
print("ASESORAI:", asistente_helix(
    "Quiero hacer un reclamo formal: me cobraron una comisión indebida hace 6 meses y todavía no me la devolvieron.",
    historial_b, cliente_id="C002", log_sesion=log_sesion
))

CONVERSACIÓN A — Cliente C001 (Carlos)

CLIENTE: Hola, soy Carlos. Tengo $2.000.000 para invertir, ¿cuánto rinde el FCI de renta fija?
ASESORAI: Hola Carlos, gracias por considerar a Helix para tu inversión. El FCI de renta fija que ofrecemos tiene un rendimiento promedio de 9.2% mensual y permite el rescate de tus fondos en un plazo de 24 a 48 horas. Esto te brinda una buena opción para obtener rendimientos atractivos con una liquidez razonable.

Si estás interesado en conocer más detalles o deseas invertir en este producto, puedo ayudarte con el proceso. ¿Te gustaría saber más sobre cómo funciona o cómo puedes comenzar a invertir con nosotros?

CLIENTE: Buenísimo. Por otro lado, ¿cuál es mi saldo actual?
ASESORAI: Hola Carlos, según nuestros registros, tu saldo actual es de $1.500.000 en pesos argentinos y $8.000 en dólares estadounidenses. ¿Necesitas ayuda con algo más o deseas realizar alguna operación con estos fondos?

CLIENTE: ¿Y si en cambio necesito algo con liquidez inmediata

In [15]:
for i, registro in enumerate(log_sesion, 1):
    print(f"--- Registro {i} ---")
    print(f"Cliente: {registro['cliente_id']}")
    print(f"Mensaje: {registro['mensaje_usuario'][:60]}...")
    print(f"FAQ match: {'Sí (' + str(registro['faq_score']) + ')' if registro['faq_match'] else 'No (' + str(registro['faq_score']) + ')'}")
    print(f"Tools llamadas: {[t['funcion'] for t in registro['tools_llamadas']] or 'ninguna'}")
    print(f"Derivado: {registro['derivado']}")
    print()

with open("conversaciones_log.json", "w", encoding="utf-8") as f:
    json.dump(log_sesion, f, ensure_ascii=False, indent=2)

print("✅ Log guardado en conversaciones_log.json")

--- Registro 1 ---
Cliente: C001
Mensaje: Hola, soy Carlos. Tengo $2.000.000 para invertir, ¿cuánto ri...
FAQ match: Sí (0.665)
Tools llamadas: ['buscar_producto']
Derivado: False

--- Registro 2 ---
Cliente: C001
Mensaje: Buenísimo. Por otro lado, ¿cuál es mi saldo actual?...
FAQ match: Sí (0.628)
Tools llamadas: ['consultar_saldo']
Derivado: False

--- Registro 3 ---
Cliente: C001
Mensaje: ¿Y si en cambio necesito algo con liquidez inmediata, por si...
FAQ match: Sí (0.734)
Tools llamadas: ['buscar_producto']
Derivado: False

--- Registro 4 ---
Cliente: C002
Mensaje: Quiero hacer un reclamo formal: me cobraron una comisión ind...
FAQ match: No (0.531)
Tools llamadas: ninguna
Derivado: True

✅ Log guardado en conversaciones_log.json


# AsesorAI Helix — Asistente Bancario Inteligente

Sistema integrado que combina historial de conversación, búsqueda semántica
en FAQs (RAG simple) y function calling sobre datos de cliente, con logging
de auditoría y derivación a un asesor humano cuando corresponde.

## Arquitectura

Usuario → Búsqueda semántica en FAQs (inyecta contexto si hay match) →
Modelo (Groq llama-3.3-70b-versatile) + historial + tools →
[si el modelo pide una función → se ejecuta y se devuelve el resultado] →
Respuesta final → Log JSON (timestamp, FAQ match, tools usadas, derivación)

## Stack

- Groq API (`llama-3.3-70b-versatile`) — compatible con SDK de OpenAI
- `sentence-transformers` (`all-MiniLM-L6-v2`) — embeddings 100% locales
- `getpass` — manejo seguro de API key

## Pruebas realizadas

**Conversación A (cliente C001, 3 turnos):** consulta de producto con FAQ
match (0.665), consulta de saldo con function calling, y consulta de
liquidez con FAQ match (0.734) — manteniendo memoria de contexto en los 3 turnos.

**Conversación B (cliente C002, 1 turno):** reclamo formal — correctamente
identificado como fuera de scope y derivado al asesor humano.

## Hallazgos y decisiones de diseño

**1. Heurística de derivación — falso positivo corregido:**
La primera versión marcaba como "derivado" cualquier respuesta que mencionara
"asesor de cuenta", incluyendo cierres de cortesía/upsell tras resolver
correctamente la consulta. Se ajustó para detectar solo frases de incapacidad
explícita ("no puedo ayudarte con...", "está fuera de mi alcance").

**2. Limitación conocida — llamadas redundantes a function calling:**
Cuando una FAQ matchea y se inyecta como contexto, el modelo a veces igual
llama a `buscar_producto` para el mismo dato, generando una llamada extra
innecesaria. Mejora futura: instruir al system prompt para que reconozca
cuándo el contexto FAQ ya es suficiente.

**3. Prompt engineering iterativo (heredado del Ej2):**
"Helix es una fintech, no un banco" — evita terminología inconsistente.

**4. Catálogo ampliado (heredado del Ej3):**
`buscar_producto` incluye entradas genéricas ("fci", "fondo") para reducir
respuestas de "producto no encontrado" ante consultas amplias.

## Limitaciones / próximos pasos

- Cálculos de rendimiento los hace el LLM directamente — en producción debería
  delegarse a una función de cálculo validada (no confiar en aritmética del modelo)
- Corpus de FAQs es chico (11 entradas) — en producción se ampliaría con
  variantes reales extraídas de tickets de soporte